In [27]:
import os, ssl

pem_path = r"C:\Users\simon.ngugi\Downloads\corp_certs.pem"
if not os.path.exists(pem_path):
    ctx = ssl.create_default_context()
    certs = ctx.get_ca_certs(binary_form=True)
    with open(pem_path, "wb") as f:
        for cert in certs:
            f.write(ssl.DER_cert_to_PEM_cert(cert).encode())

os.environ["REQUESTS_CA_BUNDLE"] = pem_path
os.environ["SSL_CERT_FILE"] = pem_path

import snowflake.connector
conn = snowflake.connector.connect(
    account="ftyokgn-cy22613", user="PHOENIX",
    password="1Leviticus.123456", database="ETL_WORKSHOP", warehouse="COMPUTE_WH",
)
cur = conn.cursor()
print("Connected")

c:\Users\simon.ngugi\Downloads\class\env\Lib\site-packages\snowflake\connector\connection\_connection.py:135: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


Connected


In [29]:
import requests, urllib3
from datetime import datetime
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

CITIES = [
    "Nairobi", "London", "New York", "Tokyo", "Sydney",
    "Lagos", "Dubai", "Berlin", "Paris", "Mumbai",
    "Cairo", "Toronto", "Singapore", "Bangkok", "Istanbul",
    "Moscow", "Seoul", "Jakarta", "Lima", "Bogota",
    "Johannesburg", "Accra", "Addis Ababa", "Casablanca", "Dar es Salaam",
    "Kampala", "Kigali", "Lusaka", "Maputo", "Kinshasa",
    "Sao Paulo", "Mexico City", "Buenos Aires", "Santiago", "Havana",
    "Amsterdam", "Stockholm", "Oslo", "Dublin", "Rome",
]

raw_data = []
for city in CITIES:
    r = requests.get(
        "https://api.openweathermap.org/data/2.5/weather",
        params={"q": city, "appid": "18890a1d8cfd1b3130af9ac578d2c1a5", "units": "metric"},
        verify=False
    )
    if r.status_code == 200:
        raw_data.append(r.json())
    else:
        print(f"  Failed: {city} ({r.status_code})")

print(f"Extracted weather for {len(raw_data)} cities")

Extracted weather for 40 cities


In [30]:
import pandas as pd
import numpy as np

rows = []
for w in raw_data:
    rows.append({
        "city":          w["name"],
        "country":       w["sys"]["country"],
        "temp_c":        round(float(w["main"]["temp"]), 1),
        "feels_like_c":  round(float(w["main"]["feels_like"]), 1),
        "humidity_pct":  int(w["main"]["humidity"]),
        "pressure_hpa":  int(w["main"]["pressure"]),
        "wind_speed_ms": round(float(w["wind"]["speed"]), 1),
        "weather":       w["weather"][0]["main"],
        "description":   w["weather"][0]["description"],
        "visibility_m":  int(w.get("visibility", 0)),
        "latitude":      round(float(w["coord"]["lat"]), 4),
        "longitude":     round(float(w["coord"]["lon"]), 4),
        "reading_time":  datetime.utcfromtimestamp(w["dt"]).strftime("%Y-%m-%d %H:%M:%S"),
        "extracted_at":  datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    })

df = pd.DataFrame(rows)
df.to_csv("weather_raw.csv", index=False)
print(f"Saved {len(df)} rows to weather_raw.csv")

Saved 40 rows to weather_raw.csv


C:\Users\simon.ngugi\AppData\Local\Temp\ipykernel_16224\2323545.py:19: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  "reading_time":  datetime.utcfromtimestamp(w["dt"]).strftime("%Y-%m-%d %H:%M:%S"),


In [31]:
weather = pd.read_csv("weather_raw.csv")
weather = weather.replace({np.nan: None})

cols = ["city", "country", "temp_c", "feels_like_c", "humidity_pct",
        "pressure_hpa", "wind_speed_ms", "weather", "description",
        "visibility_m", "latitude", "longitude", "reading_time", "extracted_at"]

data = [tuple(row[c] for c in cols) for _, row in weather.iterrows()]
ph = ", ".join(["%s"] * len(cols))

cur.execute("TRUNCATE TABLE BRONZE.WEATHER")
cur.executemany(f"INSERT INTO BRONZE.WEATHER ({', '.join(cols)}) VALUES ({ph})", data)
conn.commit()
print(f"Loaded {len(data)} rows into BRONZE.WEATHER")

Loaded 40 rows into BRONZE.WEATHER


In [32]:
now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Expire changed rows
cur.execute(f"""
    MERGE INTO SILVER.WEATHER_SCD2 tgt
    USING (
        SELECT * FROM (
            SELECT *, ROW_NUMBER() OVER (PARTITION BY city ORDER BY extracted_at DESC) AS rn
            FROM BRONZE.WEATHER
        ) WHERE rn = 1
    ) src
    ON src.city = tgt.city AND tgt.is_current = TRUE
    WHEN MATCHED AND (
        src.temp_c != tgt.temp_c
        OR src.humidity_pct != tgt.humidity_pct
        OR src.weather != tgt.weather
    ) THEN UPDATE SET
        tgt.valid_to = '{now}',
        tgt.is_current = FALSE
""")

# Insert new + changed
cur.execute(f"""
    INSERT INTO SILVER.WEATHER_SCD2
    SELECT src.city, src.country, src.temp_c, src.feels_like_c,
           src.humidity_pct, src.pressure_hpa, src.wind_speed_ms,
           src.weather, src.description, src.visibility_m,
           src.latitude, src.longitude, src.reading_time,
           src.extracted_at, '{now}', NULL, TRUE
    FROM (
        SELECT * FROM (
            SELECT *, ROW_NUMBER() OVER (PARTITION BY city ORDER BY extracted_at DESC) AS rn
            FROM BRONZE.WEATHER
        ) WHERE rn = 1
    ) src
    LEFT JOIN SILVER.WEATHER_SCD2 tgt ON src.city = tgt.city AND tgt.is_current = TRUE
    WHERE tgt.city IS NULL
""")

conn.commit()
cur.execute("SELECT COUNT(*), SUM(CASE WHEN is_current THEN 1 ELSE 0 END) FROM SILVER.WEATHER_SCD2")
total, active = cur.fetchone()
print(f"Weather SCD2: {total} total ({active} current, {total - active} historical)")

Weather SCD2: 40 total (40 current, 0 historical)
